# Setup
Please enable the 2x T4 GPU accelerator, and turn on Internet access to download HF models.

Add these datasets to the notebook (if you lack permission, ask Steven):
- https://www.kaggle.com/datasets/stevenliiem/eiv-all-heads-inference/versions/1
- https://www.kaggle.com/datasets/stevenliiem/extract-emotion2vec-embeddings-final-corpus/versions/11\
- https://www.kaggle.com/datasets/stevenliiem/corpora-cleaned/versions/1

# SER evaluation pipeline

This Kaggle notebook explores the entire tone bias from start to end. 

Run the cells in order. They follow the chronological sequence:

1. Baseline inference (EIV and emotion2vec) to test the better SER model (emotion2vec beats EIV).
2. Frozen SSL comparison (wav2vec2 and WavLM) to find the better (these two beat emotion2vec).
3. WavLM head selection (out of pca32 logistic, L2 logistic, shrinkage LDA, PCA-MLP, RBF-SVC, PLS-DA, LinearSVC, hist GBM, focal MLP).
4. Stratified unlabelled-pool inference and final evaluation.

In [ ]:
# Install shared dependencies once and confirm the GPU.
!pip -q install transformers huggingface_hub funasr modelscope soundfile scipy scikit-learn seaborn

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DATA = "/kaggle/input/datasets/stevenliiem/extract-emotion2vec-embeddings-final-corpus" # change to the correct path
CORPORA = f"{DATA}/our_speech_corpus_final/our_speech_corpus_final"
MANIFEST = f"{DATA}/our_speech_corpus_manifest_split_80_20.csv"

## 1. Baseline inference: Empathic-Insight-Voice

Run `empathic_insight_voice_all40_kaggle.py` over the cleaned corpora. Its predictions provide the first legacy SER baseline.

In [ ]:
!python /kaggle/input/datasets/stevenliiem/eiv-all-heads-inference/empathic_insight_voice_all40_kaggle.py \
  --corpora-root /kaggle/input/datasets/stevenliiem/corpora-cleaned/corpora_cleaned \
  --batch-size 24

## 2. Baseline inference: emotion2vec

Run `extract_emotion2vec_embeddings_kaggle.py` to extract emotion2vec+ utterance embeddings for the 854 human-labelled clips. These embeddings feed the same frozen-head evaluation used for the SSL models.

In [ ]:
!python {DATA}/extract_emotion2vec_embeddings_kaggle.py \
  --corpora-root {CORPORA} \
  --manifest {MANIFEST} \
  --unified {DATA}/our_speech_corpus_unified.csv \
  --output-dir /kaggle/working/emotion2vec \
  --splits train test

## 3. Frozen SSL comparison: wav2vec2 vs WavLM

Run `run_ssl_backbone_kaggle.py` to extract labelled-set embeddings and run the identical 5-fold × 10-repeat head grid for both backbones. This isolates the backbone comparison.

In [ ]:
for model, output_dir in [
    ("wav2vec2-large-robust", "/kaggle/working/ssl_wav2vec2"),
    ("wavlm-large", "/kaggle/working/ssl_wavlm"),
]:
    !python {DATA}/run_ssl_backbone_kaggle.py \
      --model {model} \
      --corpora-root {CORPORA} \
      --manifest {MANIFEST} \
      --output-dir {output_dir} \
      --run-cv \
      --folds 5 --repeats 10

## 4. WavLM head training and classifier selection

WavLM was the strongest frozen backbone. Run `train_valence_head_cv.py` to re-score the expanded head grid with the revised human labels, then validate with grouped CV to prevent lexical/template leakage.

In [ ]:
WAVLM_LABELLED = f"{DATA}/our_speech_corpus_microsoft_wavlm-large.npz"

!python {DATA}/train_valence_head_cv.py \
  --embeddings {WAVLM_LABELLED} \
  --manifest {MANIFEST} \
  --output /kaggle/working/ssl_wavlm/head_grid_new_labels.json \
  --folds 5 --repeats 10

### Group-aware validation and final choice

Re-run `train_valence_head_cv.py` with `--group-by response_text` and `--group-by question`. The harder question-grouped result is the primary development estimate. Select **balanced RBF-SVC** by macro-F1; retain **balanced L2 logistic** as the robustness check.

In [ ]:
for group_by in ["response_text", "question"]:
    !python {DATA}/train_valence_head_cv.py \
      --embeddings {WAVLM_LABELLED} \
      --manifest {MANIFEST} \
      --output /kaggle/working/ssl_wavlm/head_group_{group_by}.json \
      --folds 5 --repeats 10 \
      --group-by {group_by}

## 5. Final inference on the unlabelled pool

Run `run_ssl_backbone_kaggle.py` to extract WavLM embeddings for the 2,346 unlabelled clips, then `apply_valence_head_unlabelled.py` to fit balanced RBF-SVC and L2 logistic on all 854 labelled clips and report Corpus A and Corpus B separately (never pooled).

In [ ]:
!python {DATA}/run_ssl_backbone_kaggle.py \
  --model wavlm-large \
  --corpora-root {CORPORA} \
  --manifest {MANIFEST} \
  --splits unlabeled_pool \
  --output-dir /kaggle/working/ssl_wavlm_pool

WAVLM_POOL = "/kaggle/working/ssl_wavlm_pool/our_speech_corpus_microsoft_wavlm-large.npz"

!python {DATA}/apply_valence_head_unlabelled.py \
  --embeddings {WAVLM_LABELLED} {WAVLM_POOL} \
  --manifest {MANIFEST} \
  --candidates rbf_svc l2_logistic --balanced \
  --output-dir /kaggle/working/unlabelled_pool_preds

In [ ]:
# Print final metrics and create the only results archive.
import json
from pathlib import Path
import zipfile

working = Path("/kaggle/working")
grouped = json.loads((working / "ssl_wavlm/head_group_question.json").read_text())
for name in ["rbf_svc", "l2_logistic"]:
    result = grouped["grid"]["balanced"][name]
    print(
        f"{name}: accuracy={result['accuracy']['mean']:.3f}, "
        f"macro-F1={result['macro_f1']['mean']:.3f}, "
        f"negative recall={result['per_class']['negative']['recall']['mean']:.3f}"
    )

pool = json.loads((working / "unlabelled_pool_preds/unlabelled_pool_strata_report.json").read_text())
for name, result in pool["candidates"].items():
    print(f"\n{name}")
    for stratum, stats in result["strata"].items():
        p = stats["predicted_proportions"]
        print(f"  {stratum}: pos={p['positive']:.3f}, neu={p['neutral']:.3f}, neg={p['negative']:.3f}")

archive = working / "ser_pipeline_results.zip"
folders = ["eiv_all40_results", "emotion2vec", "ssl_wav2vec2", "ssl_wavlm", "unlabelled_pool_preds"]
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in folders:
        root = working / folder
        if root.exists():
            for file in root.rglob("*"):
                if file.is_file():
                    zf.write(file, file.relative_to(working))
print(f"\nSaved: {archive}")